# Modul Pembelajaran First-Order Logic (FOL) RKA 25


## 1. Pendahuluan

Logika proposisional hanya mengenal **fakta** utuh (`P12` = "ada pit di [1,2]"). Akibatnya, aturan yang sama harus ditulis berulang untuk setiap kotak. Contoh: aturan breeze di Wumpus World harus ditulis untuk **16 kotak**.

**First-Order Logic (FOL)** / logika predikat lebih ekspresif karena dunia dipandang terdiri dari:

| Komponen | Contoh |
|---|---|
| **Objek** (konstanta) | `John`, `Ani`, `M1` |
| **Relasi** (predikat) | `Parent(Ani, Budi)`, `King(John)` |
| **Fungsi** | `Mother(Budi)` → objek lain |
| **Variabel** | `x`, `y`, `z` |
| **Kuantor** | ∀ (untuk semua), ∃ (ada) |

Dengan FOL, aturan breeze cukup ditulis sekali:

```
∀s  Breezy(s) ⇔ ∃r Adjacent(r, s) ∧ Pit(r)
```

Pada modul ini kita mengimplementasikan dari nol:
1. Evaluasi kuantor pada model berhingga
2. **Substitusi** dan **Unifikasi**
3. **Forward Chaining** FOL
4. **Backward Chaining** FOL

Referensi: Russell & Norvig, *Artificial Intelligence: A Modern Approach*, Bab 8 dan 9.


## 2. Kuantor dan Model

### 2.1 Kuantor Universal dan Eksistensial

| Kuantor | Notasi | Arti | Pasangan alami |
|---|---|---|---|
| Universal | `∀x P(x)` | P benar untuk **setiap** objek | ⇒ (implikasi) |
| Eksistensial | `∃x P(x)` | P benar untuk **minimal satu** objek | ∧ (konjungsi) |

Kesalahan umum:
- `∀x King(x) ∧ Person(x)` artinya "semua objek adalah raja **dan** orang". Ini salah. Yang benar: `∀x King(x) ⇒ Person(x)`.
- `∃x Crown(x) ⇒ OnHead(x, John)` benar meskipun ada objek yang bukan mahkota. Yang benar: `∃x Crown(x) ∧ OnHead(x, John)`.

Hubungan De Morgan pada kuantor:
```
¬∀x P(x)  ≡  ∃x ¬P(x)
¬∃x P(x)  ≡  ∀x ¬P(x)
```

### 2.2 Mengevaluasi Kuantor pada Model Berhingga

Jika domain objek **berhingga**, `∀` bisa dihitung dengan `all()` dan `∃` dengan `any()` di Python.


In [ ]:
# Model: domain objek + interpretasi predikat (sebagai himpunan tuple)
domain = ['Ani', 'Budi', 'Citra', 'Dodi', 'Eka']

Person = {'Ani', 'Budi', 'Citra', 'Dodi', 'Eka'}
Female = {'Ani', 'Citra', 'Eka'}
Parent = {('Ani', 'Budi'), ('Ani', 'Citra'), ('Budi', 'Dodi'), ('Citra', 'Eka')}
Loves  = {('Ani', 'Budi'), ('Budi', 'Ani'), ('Citra', 'Ani'), ('Dodi', 'Ani'), ('Eka', 'Ani')}

def forall(pred):  return all(pred(x) for x in domain)
def exists(pred):  return any(pred(x) for x in domain)
def implies(a, b): return (not a) or b


In [ ]:
# ∀x Female(x) ⇒ Person(x)
print("∀x Female(x) ⇒ Person(x)        :", forall(lambda x: implies(x in Female, x in Person)))

# ∃x Parent(x, Citra)
print("∃x Parent(x, Citra)              :", exists(lambda x: (x, 'Citra') in Parent))

# ∀x ∃y Loves(x, y)   -> setiap orang mencintai seseorang
print("∀x ∃y Loves(x, y)                :", forall(lambda x: exists(lambda y: (x, y) in Loves)))

# ∃y ∀x Loves(x, y)   -> ada seseorang yang dicintai semua orang
print("∃y ∀x Loves(x, y)                :", exists(lambda y: forall(lambda x: (x, y) in Loves)))

# ∃y ∀x (x ≠ y ⇒ Loves(x, y))  -> ada seseorang yang dicintai semua orang lain
print("∃y ∀x (x≠y ⇒ Loves(x, y))        :", exists(lambda y: forall(lambda x: implies(x != y, (x, y) in Loves))))


**Urutan kuantor penting!**
- `∀x ∃y Loves(x, y)` → setiap orang punya seseorang yang ia cintai (boleh berbeda-beda).
- `∃y ∀x Loves(x, y)` → ada **satu** orang yang dicintai oleh **semua** orang (termasuk dirinya sendiri). Hasilnya False karena Ani tidak mencintai dirinya sendiri.
- Jika "dirinya sendiri" dikecualikan, Ani dicintai semua orang lain, sehingga hasilnya True.

> Pendekatan ini hanya bisa dipakai jika domain berhingga dan kecil. Untuk inferensi umum kita memakai **unifikasi** dan **chaining**.


## 3. Representasi Kalimat FOL di Python

Kita memakai konvensi AIMA:
- **Variabel**: string diawali huruf kecil → `'x'`, `'y'`, `'z1'`
- **Konstanta**: string diawali huruf kapital → `'John'`, `'M1'`
- **Atom / fungsi**: tuple `(nama, arg1, arg2, ...)` dengan nama diawali huruf kapital

| FOL | Python |
|---|---|
| `King(John)` | `('King', 'John')` |
| `Knows(John, x)` | `('Knows', 'John', 'x')` |
| `Knows(y, Mother(y))` | `('Knows', 'y', ('Mother', 'y'))` |


In [ ]:
def is_variable(x):
    return isinstance(x, str) and x[:1].islower()


def fmt(x):
    # Ubah tuple menjadi notasi FOL yang mudah dibaca
    if isinstance(x, tuple):
        return f"{x[0]}({', '.join(fmt(a) for a in x[1:])})"
    return str(x)


def fmt_theta(theta):
    if theta is None:
        return "GAGAL"
    # tampilkan nilai akhir setiap variabel (rantai substitusi diselesaikan)
    return "{" + ", ".join(f"{k}/{fmt(subst(theta, v))}" for k, v in theta.items()) + "}"


print(fmt(('Knows', 'y', ('Mother', 'y'))))
print(is_variable('x'), is_variable('John'))


## 4. Substitusi dan Unifikasi

### 4.1 Substitusi

**Substitusi** θ adalah pemetaan variabel ke term, ditulis `{x/John, y/Mother(John)}`. `SUBST(θ, α)` mengganti setiap variabel di α sesuai θ.


In [ ]:
def subst(theta, x):
    if is_variable(x):
        return subst(theta, theta[x]) if x in theta else x
    if isinstance(x, tuple):
        return tuple(subst(theta, a) for a in x)
    return x


print(fmt(subst({'x': 'John', 'y': 'Richard'}, ('Brother', 'x', 'y'))))


### 4.2 Unifikasi

**Unifikasi** mencari substitusi θ yang membuat dua kalimat menjadi **identik**:

```
UNIFY(p, q) = θ  sehingga  SUBST(θ, p) = SUBST(θ, q)
```

Algoritma (AIMA Fig. 9.1):
1. Jika `x == y` → sudah identik, kembalikan θ.
2. Jika `x` variabel → `unify_var(x, y)`. Begitu juga jika `y` variabel.
3. Jika keduanya tuple dengan panjang sama → unifikasi elemen per elemen.
4. Selain itu → **gagal**.

**Occurs check**: variabel `x` tidak boleh diunifikasi dengan term yang memuat `x` sendiri, contoh `x` dengan `Mother(x)`. Tanpa pengecekan ini akan terbentuk term tak hingga `Mother(Mother(Mother(...)))`.


In [ ]:
def unify(x, y, theta=None):
    # Kembalikan substitusi (dict) yang membuat x dan y identik, atau None jika gagal
    if theta is None:
        theta = {}
    if x == y:
        return theta
    if is_variable(x):
        return unify_var(x, y, theta)
    if is_variable(y):
        return unify_var(y, x, theta)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for xi, yi in zip(x, y):
            theta = unify(xi, yi, theta)
            if theta is None:
                return None
        return theta
    return None


def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    if is_variable(x) and x in theta:
        return unify(var, theta[x], theta)
    if occur_check(var, x, theta):
        return None
    return {**theta, var: x}


def occur_check(var, x, theta):
    if var == x:
        return True
    if is_variable(x) and x in theta:
        return occur_check(var, theta[x], theta)
    if isinstance(x, tuple):
        return any(occur_check(var, a, theta) for a in x)
    return False


### 4.3 Contoh Unifikasi (AIMA Bab 9.2.2)

Query: `Knows(John, x)` ("siapa yang dikenal John?")


In [ ]:
query = ('Knows', 'John', 'x')
kb_sentences = [
    ('Knows', 'John', 'Jane'),
    ('Knows', 'y', 'Bill'),
    ('Knows', 'y', ('Mother', 'y')),
    ('Knows', 'x', 'Elizabeth'),
]

for s in kb_sentences:
    theta = unify(query, s)
    print(f"UNIFY({fmt(query)}, {fmt(s):22}) = {fmt_theta(theta)}")
    if theta is not None:
        print(f"    hasil substitusi: {fmt(subst(theta, query))}")


Kasus terakhir **gagal** karena variabel `x` dipakai di kedua kalimat: `x` tidak bisa sekaligus bernilai `John` dan `Elizabeth`.

Padahal `∀x Knows(x, Elizabeth)` berarti *semua orang* kenal Elizabeth, termasuk John. Solusinya adalah **standardizing apart**: ganti nama variabel di salah satu kalimat agar tidak bentrok.


In [ ]:
import itertools

_counter = itertools.count()

def standardize_apart(x, mapping=None):
    # Ganti setiap variabel dengan nama baru yang unik (x -> x_7, dst.)
    if mapping is None:
        mapping = {}
    if is_variable(x):
        if x not in mapping:
            mapping[x] = f"{x}_{next(_counter)}"
        return mapping[x]
    if isinstance(x, tuple):
        return tuple(standardize_apart(a, mapping) for a in x)
    return x


s = standardize_apart(('Knows', 'x', 'Elizabeth'))
print("Setelah standardize apart:", fmt(s))
print("UNIFY =", fmt_theta(unify(query, s)))


In [ ]:
# Occurs check: x tidak bisa diunifikasi dengan Mother(x)
print(fmt_theta(unify('x', ('Mother', 'x'))))


## 5. Knowledge Base Definite Clause

Seperti pada logika proposisional, inferensi FOL yang efisien memakai **definite clause**: konjungsi atom positif ⇒ satu atom positif. Variabel dianggap terkuantifikasi universal (∀).

Kita representasikan aturan sebagai `Rule(premises, conclusion)`. Fakta adalah aturan tanpa premis.

### 5.1 Studi Kasus: Kejahatan Kolonel West (AIMA Bab 9.3.1)

> *Hukum menyatakan bahwa seorang warga Amerika yang menjual senjata kepada negara musuh adalah kriminal. Negara Nono, musuh Amerika, memiliki beberapa misil, dan semua misilnya dijual kepadanya oleh Kolonel West, seorang warga Amerika.*

| Kalimat FOL | Keterangan |
|---|---|
| `American(x) ∧ Weapon(y) ∧ Sells(x, y, z) ∧ Hostile(z) ⇒ Criminal(x)` | aturan kriminal |
| `Owns(Nono, M1)`, `Missile(M1)` | Nono punya misil M1 (dari ∃x Owns(Nono,x) ∧ Missile(x)) |
| `Missile(x) ∧ Owns(Nono, x) ⇒ Sells(West, x, Nono)` | semua misil Nono dijual West |
| `Missile(x) ⇒ Weapon(x)` | misil adalah senjata |
| `Enemy(x, America) ⇒ Hostile(x)` | musuh Amerika itu *hostile* |
| `American(West)` | West warga Amerika |
| `Enemy(Nono, America)` | Nono musuh Amerika |


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Rule:
    premises: tuple
    conclusion: tuple

    def __repr__(self):
        if not self.premises:
            return fmt(self.conclusion)
        return " ∧ ".join(fmt(p) for p in self.premises) + " ⇒ " + fmt(self.conclusion)


def fact(atom):
    return Rule((), atom)


crime_kb = [
    Rule((('American', 'x'), ('Weapon', 'y'), ('Sells', 'x', 'y', 'z'), ('Hostile', 'z')),
         ('Criminal', 'x')),
    fact(('Owns', 'Nono', 'M1')),
    fact(('Missile', 'M1')),
    Rule((('Missile', 'x'), ('Owns', 'Nono', 'x')), ('Sells', 'West', 'x', 'Nono')),
    Rule((('Missile', 'x'),), ('Weapon', 'x')),
    Rule((('Enemy', 'x', 'America'),), ('Hostile', 'x')),
    fact(('American', 'West')),
    fact(('Enemy', 'Nono', 'America')),
]

for r in crime_kb:
    print(r)


## 6. Forward Chaining FOL

### 6.1 Algoritma

Forward chaining bersifat **data-driven**: mulai dari fakta, picu aturan yang premisnya terpenuhi, tambahkan kesimpulan sebagai fakta baru, ulangi sampai tidak ada fakta baru (*fixed point*).

Untuk setiap aturan, kita cari semua substitusi θ yang membuat **semua premis** cocok dengan fakta yang ada (*pattern matching*), lalu tambahkan `SUBST(θ, conclusion)`.

Karena semua fakta adalah **ground** (tidak memiliki variabel), kita tidak perlu standardize apart pada forward chaining.


In [ ]:
def match_premises(premises, facts, theta):
    # Generator: semua θ yang membuat setiap premis cocok dengan salah satu fakta
    if not premises:
        yield theta
        return
    first, rest = premises[0], premises[1:]
    for f in facts:
        t = unify(first, f, theta)
        if t is not None:
            yield from match_premises(rest, facts, t)


def fol_fc_ask(kb, query, verbose=True):
    facts = {r.conclusion for r in kb if not r.premises}
    rules = [r for r in kb if r.premises]
    iteration = 0
    while True:
        iteration += 1
        new = set()
        for rule in rules:
            for theta in match_premises(rule.premises, facts, {}):
                inferred = subst(theta, rule.conclusion)
                if inferred not in facts and inferred not in new:
                    new.add(inferred)
                    if verbose:
                        print(f"[iterasi {iteration}] {fmt(inferred):28} dari  {rule}")
        if not new:
            break
        facts |= new
    # Jawaban: semua substitusi yang membuat query cocok dengan fakta
    answers = [unify(query, f) for f in facts]
    return [a for a in answers if a is not None]


In [ ]:
answers = fol_fc_ask(crime_kb, ('Criminal', 'x'))
print("\nJawaban Criminal(x):", [fmt_theta(a) for a in answers])


Hasil di atas sama dengan *proof tree* forward chaining di AIMA Fig. 9.4:
- **Iterasi 1**: `Sells(West, M1, Nono)`, `Weapon(M1)`, `Hostile(Nono)`
- **Iterasi 2**: `Criminal(West)`
- **Iterasi 3**: tidak ada fakta baru → *fixed point*

## 7. Backward Chaining FOL

### 7.1 Algoritma

Backward chaining bersifat **goal-driven**: mulai dari **query**, cari aturan yang kesimpulannya bisa diunifikasi dengan query, lalu buktikan premis-premisnya secara rekursif (DFS).

Algoritma (AIMA Fig. 9.6) terdiri dari dua fungsi yang saling memanggil:
- **`bc_or(goal)`**: goal bisa dibuktikan oleh aturan mana pun (**OR**).
- **`bc_and(goals)`**: semua sub-goal harus terbukti (**AND**).

Berbeda dengan forward chaining, di sini aturan **wajib di-standardize apart** setiap kali dipakai, karena variabel dari goal dan aturan bisa bentrok.


In [ ]:
def bc_or(kb, goal, theta, depth, verbose):
    for rule in kb:
        mapping = {}
        conclusion = standardize_apart(rule.conclusion, mapping)
        premises = tuple(standardize_apart(p, mapping) for p in rule.premises)
        t = unify(goal, conclusion, theta)
        if t is not None:
            if verbose:
                print("  " * depth + f"goal {fmt(subst(theta, goal))}  ←  {rule}")
            yield from bc_and(kb, premises, t, depth + 1, verbose)


def bc_and(kb, goals, theta, depth, verbose):
    if not goals:
        yield theta
        return
    first, rest = goals[0], goals[1:]
    for t1 in bc_or(kb, subst(theta, first), theta, depth, verbose):
        yield from bc_and(kb, rest, t1, depth, verbose)


def fol_bc_ask(kb, query, verbose=False):
    # Kembalikan semua jawaban unik (query yang sudah disubstitusi)
    results = []
    for theta in bc_or(kb, query, {}, 0, verbose):
        ans = subst(theta, query)
        if ans not in results:
            results.append(ans)
    return results


In [ ]:
answers = fol_bc_ask(crime_kb, ('Criminal', 'x'), verbose=True)
print("\nJawaban:", [fmt(a) for a in answers])


Jejak di atas menunjukkan pohon pembuktian *backward* (AIMA Fig. 9.7): untuk membuktikan `Criminal(x)`, agen membuktikan `American(x)` → `x = West`, lalu `Weapon(y)`, `Sells(West, y, z)`, dan `Hostile(z)` secara berurutan.

### 7.2 Studi Kasus: Silsilah Keluarga

Backward chaining juga bisa menjawab query dengan banyak jawaban, termasuk aturan **rekursif** seperti `Ancestor`.

```
       Ani
      /    \
   Budi    Citra
    |        |
   Dodi     Eka
```


In [ ]:
family_kb = [
    fact(('Parent', 'Ani', 'Budi')),
    fact(('Parent', 'Ani', 'Citra')),
    fact(('Parent', 'Budi', 'Dodi')),
    fact(('Parent', 'Citra', 'Eka')),
    # Ancestor(x, y) ⇐ Parent(x, y)
    Rule((('Parent', 'x', 'y'),), ('Ancestor', 'x', 'y')),
    # Ancestor(x, z) ⇐ Parent(x, y) ∧ Ancestor(y, z)
    Rule((('Parent', 'x', 'y'), ('Ancestor', 'y', 'z')), ('Ancestor', 'x', 'z')),
]

print("Keturunan Ani     :", [fmt(a) for a in fol_bc_ask(family_kb, ('Ancestor', 'Ani', 'w'))])
print("Leluhur Eka       :", [fmt(a) for a in fol_bc_ask(family_kb, ('Ancestor', 'w', 'Eka'))])
print("Ancestor(Budi,Eka):", fol_bc_ask(family_kb, ('Ancestor', 'Budi', 'Eka')))


List kosong `[]` berarti query **tidak bisa dibuktikan** dari KB.

> ⚠️ Backward chaining berbasis DFS bisa **terjebak loop tak hingga** pada aturan *left-recursive*, misalnya `Ancestor(x, z) ⇐ Ancestor(x, y) ∧ Parent(y, z)`. Karena itu, letakkan predikat dasar (`Parent`) di depan predikat rekursif.

## 8. Perbandingan dan Ringkasan

| Aspek | Forward Chaining | Backward Chaining |
|---|---|---|
| Arah | Data-driven (fakta → kesimpulan) | Goal-driven (query → fakta) |
| Strategi | Iterasi sampai *fixed point* | DFS rekursif |
| Standardize apart | Tidak perlu (fakta ground) | Wajib setiap aturan dipakai |
| Kelebihan | Menurunkan **semua** konsekuensi KB | Hanya mengerjakan yang relevan dengan query |
| Kekurangan | Bisa menurunkan banyak fakta tidak relevan | Bisa loop & pekerjaan berulang |
| Contoh penerapan | Sistem produksi, *deductive database* | Prolog, sistem pakar |

| Propositional Logic | First-Order Logic |
|---|---|
| Fakta utuh (`P12`) | Objek, relasi, fungsi |
| Tidak ada variabel | Variabel + kuantor ∀, ∃ |
| Pencocokan simbol identik | **Unifikasi** |
| Entailment *decidable* | Entailment *semi-decidable* |


## 9. Latihan Soal (PR)
<font color="blue">Latihan 7</font>


### PR 1 — Unifikasi

Tentukan hasil unifikasi berikut **secara manual** terlebih dahulu, lalu cek dengan fungsi `unify`. Tuliskan alasan jika unifikasi gagal.

| No | p | q |
|---|---|---|
| a | `Older(Father(y), y)` | `Older(Father(x), John)` |
| b | `Knows(Father(y), y)` | `Knows(x, x)` |
| c | `P(A, y, F(y))` | `P(x, B, z)` |
| d | `Q(x, G(x))` | `Q(G(y), y)` |


In [ ]:
pairs = [
    (('Older', ('Father', 'y'), 'y'), ('Older', ('Father', 'x'), 'John')),
    (_____, _____),   # b
    (_____, _____),   # c
    (_____, _____),   # d
]

for p, q in pairs:
    print(f"UNIFY({fmt(p)}, {fmt(q)}) = {fmt_theta(unify(p, q))}")


**Analisis (tuliskan di sini):**
- a: ...
- b: ...
- c: ...
- d: ...


### PR 2 — Silsilah Keluarga

Diberikan silsilah keluarga berikut:

```
          Hasan (L) ── Siti (P)
          /                  \
     Rudi (L) ── Maya (P)    Lina (P) ── Joko (L)
       /     \                   |
  Tono (L)  Wati (P)          Sari (P)
```

**Tugas:** Lengkapi aturan berikut dalam bentuk definite clause:
1. `Mother(x, y)`: x adalah ibu dari y
2. `Grandparent(x, z)`: x adalah kakek/nenek dari z
3. `Grandmother(x, z)`: x adalah nenek dari z
4. `Ancestor(x, z)`: x adalah leluhur z (rekursif)

Kemudian jawab query menggunakan **forward chaining** dan **backward chaining**:
- Siapa saja cucu dari Hasan?
- Siapa nenek dari Sari?
- Siapa saja keturunan Siti?


In [ ]:
pr_kb = [
    fact(('Parent', 'Hasan', 'Rudi')), fact(('Parent', 'Siti', 'Rudi')),
    fact(('Parent', 'Hasan', 'Lina')), fact(('Parent', 'Siti', 'Lina')),
    fact(('Parent', 'Rudi', 'Tono')),  fact(('Parent', 'Maya', 'Tono')),
    fact(('Parent', 'Rudi', 'Wati')),  fact(('Parent', 'Maya', 'Wati')),
    fact(('Parent', 'Lina', 'Sari')),  fact(('Parent', 'Joko', 'Sari')),
    fact(('Female', 'Siti')), fact(('Female', 'Maya')), fact(('Female', 'Lina')),
    fact(('Female', 'Wati')), fact(('Female', 'Sari')),

    # 1. Mother(x, y) ⇐ Parent(x, y) ∧ Female(x)
    Rule((('Parent', 'x', 'y'), ('Female', 'x')), ('Mother', 'x', 'y')),

    # 2. Grandparent(x, z) ⇐ ...
    Rule(_____, ('Grandparent', 'x', 'z')),

    # 3. Grandmother(x, z) ⇐ ...
    Rule(_____, ('Grandmother', 'x', 'z')),

    # 4. Ancestor(x, z) ⇐ ...   (dua aturan: kasus dasar dan rekursif)
    Rule(_____, ('Ancestor', 'x', 'z')),
    Rule(_____, ('Ancestor', 'x', 'z')),
]


In [ ]:
# Forward chaining
print("Cucu Hasan (FC):", [fmt_theta(t) for t in fol_fc_ask(pr_kb, ('Grandparent', 'Hasan', 'w'), verbose=False)])

# Backward chaining
print("Cucu Hasan      :", [fmt(a) for a in fol_bc_ask(pr_kb, ('Grandparent', 'Hasan', 'w'))])
print("Nenek Sari      :", [fmt(a) for a in fol_bc_ask(pr_kb, ('Grandmother', 'w', 'Sari'))])
print("Keturunan Siti  :", [fmt(a) for a in fol_bc_ask(pr_kb, ('Ancestor', 'Siti', 'w'))])


**Pertanyaan analisis:**
1. Mengapa backward chaining perlu *standardize apart*, sedangkan forward chaining tidak?
2. Apa yang terjadi jika aturan rekursif `Ancestor` ditulis `Ancestor(x, z) ⇐ Ancestor(x, y) ∧ Parent(y, z)`? Coba dan jelaskan.

**Jawaban:** ...
